In [1]:
from google.adk.agents import Agent, LlmAgent 
from google.adk.models.lite_llm import LiteLlm 
import os 
from utils.tools import check_warehouse_availability, reserve_warehouse_items 

/home/k/AI-Engineering-Bootcamp/01-ai-engineering-bootcamp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### A2A Client

In [2]:
import uuid 

import httpx 
from a2a.client import A2ACardResolver, A2AClient 
from a2a.types import (
    AgentCard,  
    Message, 
    MessageSendParams, 
    Part, 
    Role, 
    SendMessageRequest, 
    TextPart, 
)

PUBLIC_AGENT_CARD_PATH = "/.well-known/agent.json" 
BASE_URL = "http://localhost:10001"

In [3]:
async with httpx.AsyncClient() as httpx_client: 
    # Initialize A2ACardResolver 
    resolver = A2ACardResolver( 
        httpx_client=httpx_client, 
        base_url = BASE_URL,  
    )

    try: 
        print( 
            f"Fetching public agend card from {BASE_URL}{PUBLIC_AGENT_CARD_PATH}" 
        ) 
        _public_card = await resolver.get_agent_card() 
        print("Fetched public agend card") 
        print(_public_card.model_dump_json(indent=2)) 

        final_agent_card_to_use = _public_card 


    except Exception as e:  
        print(f"Error fetching public agent card: {e}")  
        raise RuntimeError("Failed to fetch public agent card")  



Fetching public agend card from http://localhost:10001/.well-known/agent.json
Fetched public agend card
{
  "additionalInterfaces": null,
  "capabilities": {
    "extensions": null,
    "pushNotifications": null,
    "stateTransitionHistory": null,
    "streaming": true
  },
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [
    "text"
  ],
  "description": "An agent that can check the availability of items in the warehouses and reserve them.",
  "documentationUrl": null,
  "iconUrl": null,
  "name": "Warehouse Manager Agent",
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "provider": null,
  "security": null,
  "securitySchemes": null,
  "signatures": null,
  "skills": [
    {
      "description": "Check the availability of items in the warehouses",
      "examples": [
        "What is the availability of the item 123?"
      ],
      "id": "ABC",
      "inputModes": null,
      "name": "Check Availability",
      "outputModes": null,
      "securit

In [6]:
async with httpx.AsyncClient() as httpx_client: 
    # Initialize A2ACardResolver 
    resolver = A2ACardResolver( 
        httpx_client=httpx_client, 
        base_url = BASE_URL,  
    )

    client = A2AClient( 
        httpx_client=httpx_client, agent_card=final_agent_card_to_use,  
    ) 
    print("A2AClient initialized")  

    message_id = str(uuid.uuid4())  
    message_payload = Message( 
        role=Role.user, 
        messageId=message_id, 
        parts=[Part(root=TextPart(text="Hello, how are you?"))], 
    ) 
    id = str(uuid.uuid4()) 
    request = SendMessageRequest( 
        id=id, 
        params=MessageSendParams(  
            message=message_payload, 
        ) 
    ) 
    print("Sending message") 

    response = await client.send_message(request) 
    print("Response:") 
    print(response.model_dump_json(indent=2))  

/tmp/ipykernel_14038/46606894.py:8: DeprecationWarning: A2AClient is deprecated and will be removed in a future version. Use ClientFactory to create a client with a JSON-RPC transport.
  client = A2AClient(


A2AClient initialized
Sending message
Response:
{
  "error": {
    "code": -32603,
    "data": null,
    "message": "litellm.BadRequestError: OpenAIException - The project you are requesting has been archived and is no longer accessible"
  },
  "id": null,
  "jsonrpc": "2.0"
}
